# RF-RFE feature-selection reproduction (Linux)

This notebook reproduces the Random-Forest Recursive Feature Elimination workflow from `Model_Performance_3_10.ipynb` using the Linux Parquet feature partitions. It keeps the released computation unchanged: the fixed eight-cell test exclusion, all 65 engineered features, global `[::8]` training-row sampling, training-only `RobustScaler`, 30-tree random forests, and RFE `65 -> 50 -> 35 -> 20 -> 5`.

The RF-RFE calculation is CPU-only. ROCm/TensorFlow is not imported and the GPU is not used. The released notebook reported 7,523,137 training rows (940,393 rows after `[::8]`); the regenerated Linux partitions currently contain 7,693,642 training rows (961,706 after `[::8]`). The algorithm and feature order are preserved, but this input-row difference means exact reproduction of the historical five-feature result is not guaranteed without the unavailable original `xCC.pkl/yCC.pkl`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name not in {'ML performance', 'ML-performance'}:
    for folder_name in ('ML-performance', 'ML performance'):
        candidate = NOTEBOOK_DIR / 'VRE_Trustability' / folder_name
        if candidate.is_dir():
            NOTEBOOK_DIR = candidate
            break
WORKSPACE_ROOT = NOTEBOOK_DIR.parents[1]
FEATURE_ROOT = WORKSPACE_ROOT / 'data' / 'processed' / 'features_linux_v1'
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
ARTIFACT_BASE = WORKSPACE_ROOT / 'artifacts' / 'rf_rfe_reproduction'
ARTIFACT_ROOT = ARTIFACT_BASE / RUN_TIMESTAMP
sys.path.insert(0, str(NOTEBOOK_DIR))

from functions import linux_rf_rfe_pipeline as pipeline
importlib.reload(pipeline)

print('Features:', FEATURE_ROOT)
print('Run timestamp (UTC):', RUN_TIMESTAMP)
print('Artifacts:', ARTIFACT_ROOT)

## Released RF-RFE configuration

`random_state=None` is deliberate: the released notebook did not fix a random seed. Consequently, Random Forest bootstrap samples and possibly the selected features can vary across runs. Set an integer only for a new deterministic experiment; doing so is not the literal released configuration.

In [ ]:
config = pipeline.RFRFEConfig(
    sample_stride=8,
    n_estimators=30,
    n_jobs=32,
    n_features_to_select=5,
    step=15,
    rfe_verbose=2,
    importance_verbose=2,
    random_state=None,
)

plan = pipeline.describe_plan(FEATURE_ROOT, config)
display(pd.Series(plan).drop(labels=['train_cells']))
print('Train cells:', plan['train_cells'])

## Execute RF-RFE

This is the full paper-style selection, not a smoke test. With the current dataset it samples about 962k rows and fits six 30-tree forests in total: four elimination fits plus RFE's final five-feature estimator, followed by the separately fitted selected-feature importance model from the released notebook. A `run_status.json` file records the current stage if execution is interrupted. Rerun the initialization cell before starting again so a new timestamped artifact directory is used.

In [ ]:
RUN_RF_RFE = True

if RUN_RF_RFE:
    result = pipeline.run_paper_rf_rfe(
        feature_root=FEATURE_ROOT,
        artifact_dir=ARTIFACT_ROOT,
        config=config,
    )
    print('Selected features:', result['selected_features'])
    print('Durations:', result['durations'])
else:
    result = None
    print('RF-RFE was not started. Set RUN_RF_RFE = True when ready.')

## Inspect saved ranking

The ranking table preserves all 65 feature names, the RFE support mask/rank, the importance from RFE's final estimator, and the importance from the separately fitted post-RFE forest used by the released notebook.

In [ ]:
if result is not None:
    ranking = pd.read_csv(ARTIFACT_ROOT / 'feature_ranking.csv')
    display(ranking.sort_values(['ranking', 'feature']).reset_index(drop=True))
    display(ranking.loc[ranking['selected']].reset_index(drop=True))
    run_metadata = json.loads((ARTIFACT_ROOT / 'run.json').read_text())
    print('Saved:', ARTIFACT_ROOT)